In [1]:
from transformers import BartTokenizer
from datasets import Dataset
from transformers import BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
from scripts.Utils import TimexNorm_Utils
from scripts.Reader import obtain_dataset

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
utils = TimexNorm_Utils(tokenizer)

In [2]:
tokenizer.add_special_tokens({
  "additional_special_tokens": ["<timex","type=DATE>","type=TIME>",
                                "type=DURATION>","type=SET>","</timex>", "<sep>"]
})
model.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


BartScaledWordEmbedding(50272, 768, padding_idx=1)

In [3]:
datasets = obtain_dataset("TempEval3", "normalised")

Generating train split: 0 examples [00:00, ? examples/s]

d:\GeoTKG\scripts\Reader.py:660: SyntaxWarning: invalid escape sequence '\T'
  article = TimeMLReader.test("rawdata\TempEval3\Training\TE3-Silver-data-0\AFP_ENG_19970402.0207.tml")


FileNotFoundError: Unable to find 'D:/GeoTKG\cleandata\normalised\TempEval3\eval.json'

In [ ]:
datasets

DatasetDict({
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 20
    })
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 2432
    })
    eval: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 250
    })
})

In [ ]:
datasets = utils.tokenize_datasets(datasets)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/2432 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results/TimeNormBart",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=2,
    predict_with_generate=True,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["eval"],
    data_collator=data_collator,
    #compute_metrics=utils.compute_metrics,
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.932600
1000,0.180800
1500,0.116600
2000,0.083600


d:\GeoTKG\venv\Lib\site-packages\transformers\modeling_utils.py:3854: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=2280, training_loss=0.2963877083962424, metrics={'train_runtime': 541.0092, 'train_samples_per_second': 67.43, 'train_steps_per_second': 4.214, 'total_flos': 1.11215929982976e+16, 'train_loss': 0.2963877083962424, 'epoch': 15.0})

In [ ]:
trainer.save_model("./results/TimeNormBart")
tokenizer.save_pretrained("./results/TimeNormBart")

('./results/TimeNormBart\\tokenizer_config.json',
 './results/TimeNormBart\\special_tokens_map.json',
 './results/TimeNormBart\\vocab.json',
 './results/TimeNormBart\\merges.txt',
 './results/TimeNormBart\\added_tokens.json')

In [ ]:
predictions_output = trainer.predict(datasets["test"])
# Returns a namedtuple with:
#  - predictions_output.predictions: raw token‑ID logits or IDs (depending on config)
#  - predictions_output.label_ids: the gold token IDs

# 4) Decode predictions into strings
decoded_preds = tokenizer.batch_decode(
    predictions_output.predictions, 
    skip_special_tokens=True
)
decoded_labels = tokenizer.batch_decode(
    predictions_output.label_ids, 
    skip_special_tokens=True
)

for p, t in zip(decoded_preds, decoded_labels):
    print(f"P: {p} | T: {t}")

# 5) Compute a simple exact‑match accuracy
exact_match = sum(p == t for p, t in zip(decoded_preds, decoded_labels)) / len(decoded_labels)
print(f"Exact‑match accuracy: {exact_match:.2%}")

P: P1MPXX2013-W4720132013 -03- | T: P1MP10Y2013-W112013-03-2220092010XXXX-WIXXXX-WI
P: 19532013-03-22195319571958 | T: 19532013-03-201953195719581995P100DXXXX-XX-XX
P: PAST_REF2010-05PXX2013-03-18TA | T: PAST_REF2010-05PAST_REF2013-03-22TAF
P: 2013-03-22P1W2012-01-03 | T: 2013-03-23T15:00P1W2013-03-22
P: 2013-1020032012P18M20032010PRESENT | T: 2012-102003P12MP18M2003P1DE2011PRESENT_REF2008
P: 2013-03-222013-10-31 | T: 2013-03-222013-03-21
P: P10M | T: P5Y
P: 2013-SU2013-03-222013PAST_REFP | T: 2012-SU2013-03-212012-07PXD2012-XX-XX2012-082013-
P: 2013-W17 | T: 2013-W12
P: 2013-03-22P4W2012-04-072013- | T: 2013-03-22P4W2013-04-072013-03-222013
P: 2013-03-222010PT1MP2M | T: 2013-03-222010PXMP4Y
P: 2013-03P1M2013-10PXY20022011 | T: 2013-03XXXX-XX-XXTMO2013-02PXY200920112013-02-28P
P: 2005201020082013-W432007 | T: 2012201020082013-W122007
P: 2013-03-2120072013-05 | T: 2013-03-2120072012-05
P: PT1H2013-01-012011P90YPT24 | T: PT1H2011-01-XX2011P90YPT24HPRESENT_REF19PRESENT_
P: digitalPRESENT_